# EXPT NO: 9 — TRANSFORMER-BASED LLM CHATBOT USING STREAMLIT OR GRADIO

**Name:** NIVASRI T  
**Roll No:** 24BAD081  
**Date:** 

---
## AIM
To develop an interactive chatbot using a pre-trained Transformer-based LLM and deploy it using Gradio.

## SOFTWARE REQUIREMENTS
Python, PyTorch, Hugging Face Transformers, Gradio, Google Colab, GitHub

> **Before running:** in Colab go to **Runtime → Change runtime type → T4 GPU** (it also works on CPU, just slower).  
> Model used: **Qwen2.5-0.5B-Instruct**, a small open instruction-tuned Transformer LLM that needs no login or access token.

In [ ]:
# Install required libraries (takes ~1 minute on Colab)
!pip install -q transformers accelerate gradio

In [ ]:
# ---------------------------------------------------------------
# Experiment 9 : Transformer-based LLM Chatbot (Gradio)
# NAME    : NIVASRI T
# ROLL NO : 24BAD081
# ---------------------------------------------------------------
import time
import torch
import pandas as pd
import transformers
from transformers import AutoTokenizer, AutoModelForCausalLM, set_seed

print("Name    : NIVASRI T")
print("Roll No : 24BAD081")
print("PyTorch      :", torch.__version__)
print("Transformers :", transformers.__version__)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Running on   :", DEVICE.upper())

---
## PART A: LOADING AND CONFIGURING A PRE-TRAINED LLM

In [ ]:
# ---- A.1 - A.2 Select a pre-trained LLM, load model + tokenizer ----
# Roll No: 24BAD081 | Name: NIVASRI T
MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"     # decoder-only Transformer, ~0.5 billion parameters

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16 if DEVICE == "cuda" else torch.float32,
).to(DEVICE)
model.eval()

n_params = sum(p.numel() for p in model.parameters())
cfg = model.config
print(f"Model            : {MODEL_NAME}")
print(f"Parameters       : {n_params/1e6:.0f} million")
print(f"Transformer layers: {cfg.num_hidden_layers}")
print(f"Attention heads  : {cfg.num_attention_heads}")
print(f"Hidden size      : {cfg.hidden_size}")
print(f"Vocabulary size  : {cfg.vocab_size}")

In [ ]:
# ---- A.3 Prepare and tokenize sample user prompts ----
sample_prompts = [
    "What is machine learning?",
    "Explain the Transformer architecture in two sentences.",
    "Write a short poem about Coimbatore.",
]

for p in sample_prompts:
    ids = tokenizer(p)["input_ids"]
    print("Prompt    :", p)
    print("Tokens    :", tokenizer.convert_ids_to_tokens(ids))
    print("Token IDs :", ids)
    print("Num tokens:", len(ids), "\n")

In [ ]:
# ---- A.4 Generate responses with configurable temperature and max output length ----
def generate_reply(messages, temperature=0.7, max_new_tokens=150, top_p=0.9):
    """messages = list of {'role': ..., 'content': ...} dicts. Returns the model's reply text."""
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors="pt").to(DEVICE)

    do_sample = temperature > 0
    gen_kwargs = dict(max_new_tokens=int(max_new_tokens),
                      pad_token_id=tokenizer.eos_token_id)
    if do_sample:
        gen_kwargs.update(do_sample=True, temperature=float(temperature), top_p=top_p)
    else:
        gen_kwargs.update(do_sample=False)           # greedy decoding

    with torch.no_grad():
        output_ids = model.generate(**inputs, **gen_kwargs)

    new_tokens = output_ids[0][inputs["input_ids"].shape[1]:]     # drop the prompt part
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

set_seed(81)
for p in sample_prompts:
    reply = generate_reply([{"role": "user", "content": p}], temperature=0.7, max_new_tokens=120)
    print("USER :", p)
    print("BOT  :", reply, "\n" + "-" * 70)

In [ ]:
# ---- A.5 Examine the effect of temperature and max output length ----
prompt = [{"role": "user", "content": "Give me one creative name for a coffee shop and explain it."}]

print("=== Effect of TEMPERATURE (max_new_tokens = 60) ===")
for t in [0.0, 0.5, 1.0, 1.5]:
    set_seed(81)
    print(f"\n[temperature = {t}]")
    print(generate_reply(prompt, temperature=t, max_new_tokens=60))

print("\n\n=== Effect of MAX OUTPUT LENGTH (temperature = 0.7) ===")
for m in [20, 60, 150]:
    set_seed(81)
    out = generate_reply(prompt, temperature=0.7, max_new_tokens=m)
    print(f"\n[max_new_tokens = {m}]  ->  {len(tokenizer(out)['input_ids'])} tokens generated")
    print(out)

---
## PART B: IMPLEMENTING CHATBOT FUNCTIONALITY

In [ ]:
# ---- B.1 - B.3 Chatbot function with system prompt and conversation history ----
# Roll No: 24BAD081 | Name: NIVASRI T
SYSTEM_PROMPT = "You are a helpful, concise AI assistant created for the Deep Learning lab by NIVASRI T (24BAD081)."

class Chatbot:
    def __init__(self, system_prompt=SYSTEM_PROMPT, max_turns=6):
        self.system_prompt = system_prompt
        self.max_turns = max_turns            # keep only the latest N turns to limit context length
        self.history = []                     # list of {'role', 'content'}

    def ask(self, user_message, temperature=0.7, max_new_tokens=150):
        self.history.append({"role": "user", "content": user_message})
        recent = self.history[-2 * self.max_turns:]
        messages = [{"role": "system", "content": self.system_prompt}] + recent
        reply = generate_reply(messages, temperature, max_new_tokens)
        self.history.append({"role": "assistant", "content": reply})
        return reply

    def reset(self):
        self.history = []

bot = Chatbot()
print("Chatbot ready.")

In [ ]:
# ---- B.4 Test the chatbot: multi-turn conversation (history is remembered) ----
set_seed(81)
conversation = [
    "Hi! My name is Nivasri and I study AI and Data Science.",
    "What is my name?",
    "Suggest two project ideas related to what I study.",
    "Which of those two would be easier for a beginner?",
]
for q in conversation:
    print("USER :", q)
    print("BOT  :", bot.ask(q, temperature=0.5, max_new_tokens=120), "\n")

print("Turns stored in history:", len(bot.history) // 2)

In [ ]:
# ---- B.5 Same question WITHOUT history (shows why conversation memory matters) ----
fresh_bot = Chatbot()
set_seed(81)
print("USER : What is my name?")
print("BOT  :", fresh_bot.ask("What is my name?", temperature=0.5, max_new_tokens=60))

In [ ]:
# ---- B.6 Test with different kinds of queries and observe response quality ----
test_queries = {
    "Factual"      : "What is the capital of Tamil Nadu?",
    "Explanation"  : "Explain overfitting in simple words.",
    "Maths"        : "What is 17 multiplied by 23?",
    "Code"         : "Write a Python function to reverse a string.",
    "Creative"     : "Write a two-line motivational quote for students.",
}
bot.reset()
for kind, q in test_queries.items():
    set_seed(81)
    bot.reset()
    print(f"[{kind}] USER: {q}")
    print("BOT:", bot.ask(q, temperature=0.3, max_new_tokens=130), "\n" + "-" * 70)

---
## PART C: DEVELOPING AN INTERACTIVE INTERFACE (GRADIO)
**Gradio** is chosen because it needs very little code, works directly inside Colab, and gives a ready-made chat layout plus a public share link.

In [ ]:
# ---- C.1 - C.4 Gradio chat interface connected to the LLM ----
# Roll No: 24BAD081 | Name: NIVASRI T
import gradio as gr
print("Gradio version:", gr.__version__)

def _to_messages(history):
    """Accept Gradio history in either 'messages' (dicts) or older 'tuples' format."""
    msgs = []
    for item in history:
        if isinstance(item, dict):
            content = item["content"]
            if isinstance(content, list):          # newer Gradio may wrap text in parts
                content = "".join(c.get("text", "") if isinstance(c, dict) else str(c) for c in content)
            msgs.append({"role": item["role"], "content": content})
        else:
            user, assistant = item
            msgs.append({"role": "user", "content": user})
            if assistant:
                msgs.append({"role": "assistant", "content": assistant})
    return msgs

def chat_fn(message, history, system_prompt, temperature, max_new_tokens):
    msgs = [{"role": "system", "content": system_prompt}] + _to_messages(history)[-12:]
    msgs.append({"role": "user", "content": message})
    return generate_reply(msgs, temperature, max_new_tokens)

demo = gr.ChatInterface(
    fn=chat_fn,
    additional_inputs=[
        gr.Textbox(SYSTEM_PROMPT, label="System prompt"),
        gr.Slider(0.0, 1.5, value=0.7, step=0.1, label="Temperature"),
        gr.Slider(20, 400, value=150, step=10, label="Max new tokens"),
    ],
    title="LLM Chatbot - Qwen2.5-0.5B-Instruct  |  NIVASRI T (24BAD081)",
    description="Experiment 9: Transformer-based chatbot built with Hugging Face Transformers + Gradio.",
    examples=[["What is a Transformer model?"], ["Explain gradient descent simply."], ["Tell me a fun fact about space."]],
)

# share=True creates a temporary public link; the chat window also appears below in Colab
demo.launch(share=True, debug=False)

**How to test the interface (take screenshots):**
1. Type a query in the text box and press **Enter** / **Submit**.
2. Try different inputs (factual, code, creative, follow-up questions).
3. Change **Temperature** and **Max new tokens** in the *Additional Inputs* panel and compare answers.
4. Use the clear/retry buttons to confirm the chat layout works.

In [ ]:
# Stop the server once testing is finished
demo.close()

---
## PART D: APPLICATION TESTING AND DEPLOYMENT

In [ ]:
# ---- D.1 - D.2 Response latency and throughput ----
# Roll No: 24BAD081 | Name: NIVASRI T
latency_prompts = [
    "What is AI?",
    "Explain how a neural network learns, in detail.",
    "Write a Python program to find the factorial of a number.",
]
rows = []
for p in latency_prompts:
    for m in [50, 150]:
        set_seed(81)
        start = time.time()
        out = generate_reply([{"role": "user", "content": p}], temperature=0.7, max_new_tokens=m)
        secs = time.time() - start
        n_tok = len(tokenizer(out)["input_ids"])
        rows.append({"Prompt": p[:40], "max_new_tokens": m, "Tokens generated": n_tok,
                     "Latency (s)": round(secs, 2), "Tokens/sec": round(n_tok / secs, 1)})

latency_df = pd.DataFrame(rows)
latency_df

In [ ]:
# ---- D.3 Observe limitations: hallucination test ----
# Questions about things that do NOT exist - a reliable model should say it does not know.
hallu_tests = [
    "Who won the 2031 FIFA World Cup?",
    "Summarize the novel 'The Silver Orchids of Kovaipudur' by Dr. Anand Rangaswamy.",
    "What is the phone number of Nivasri T from Kumaraguru College of Technology?",
    "What is the exact population of Coimbatore today, to the last digit?",
]
for q in hallu_tests:
    set_seed(81)
    print("USER :", q)
    print("BOT  :", generate_reply([{"role": "user", "content": q}], temperature=0.3, max_new_tokens=100))
    print("-" * 70)

**Observation guide for the hallucination test:** if the bot confidently gives a winner for a 2031 tournament, summarises a book that does not exist, or invents a phone number, that is a *hallucination* — fluent but fabricated output. Record which prompts produced hallucinations in your observations below.

In [ ]:
# ---- D.4 Organise the source code: write app.py, requirements.txt, README.md ----
import os
os.makedirs("llm_chatbot_24BAD081", exist_ok=True)

app_code = '''# Experiment 9 - Transformer-based LLM Chatbot (Gradio)
# NAME: NIVASRI T | ROLL NO: 24BAD081
import torch
import gradio as gr
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
SYSTEM_PROMPT = "You are a helpful, concise AI assistant created by NIVASRI T (24BAD081)."

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, torch_dtype=torch.float16 if DEVICE == "cuda" else torch.float32
).to(DEVICE)
model.eval()


def generate_reply(messages, temperature=0.7, max_new_tokens=150, top_p=0.9):
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors="pt").to(DEVICE)
    kwargs = dict(max_new_tokens=int(max_new_tokens), pad_token_id=tokenizer.eos_token_id)
    if temperature > 0:
        kwargs.update(do_sample=True, temperature=float(temperature), top_p=top_p)
    else:
        kwargs.update(do_sample=False)
    with torch.no_grad():
        out = model.generate(**inputs, **kwargs)
    return tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()


def to_messages(history):
    msgs = []
    for item in history:
        if isinstance(item, dict):
            content = item["content"]
            if isinstance(content, list):
                content = "".join(c.get("text", "") if isinstance(c, dict) else str(c) for c in content)
            msgs.append({"role": item["role"], "content": content})
        else:
            user, assistant = item
            msgs.append({"role": "user", "content": user})
            if assistant:
                msgs.append({"role": "assistant", "content": assistant})
    return msgs


def chat_fn(message, history, system_prompt, temperature, max_new_tokens):
    msgs = [{"role": "system", "content": system_prompt}] + to_messages(history)[-12:]
    msgs.append({"role": "user", "content": message})
    return generate_reply(msgs, temperature, max_new_tokens)


demo = gr.ChatInterface(
    fn=chat_fn,
    additional_inputs=[
        gr.Textbox(SYSTEM_PROMPT, label="System prompt"),
        gr.Slider(0.0, 1.5, value=0.7, step=0.1, label="Temperature"),
        gr.Slider(20, 400, value=150, step=10, label="Max new tokens"),
    ],
    title="LLM Chatbot - Qwen2.5-0.5B-Instruct | NIVASRI T (24BAD081)",
    description="Experiment 9: Transformer-based chatbot using Hugging Face Transformers + Gradio.",
)

if __name__ == "__main__":
    demo.launch()
'''

requirements = "transformers\ntorch\naccelerate\ngradio\n"

readme = '''---
title: LLM Chatbot 24BAD081
emoji: 🤖
colorFrom: blue
colorTo: purple
sdk: gradio
app_file: app.py
pinned: false
---

# Transformer-based LLM Chatbot (Experiment 9)

**Name:** NIVASRI T  |  **Roll No:** 24BAD081

A chatbot built with the pre-trained **Qwen2.5-0.5B-Instruct** Transformer model (Hugging Face Transformers)
and a **Gradio** chat interface. Temperature and maximum output length can be changed from the UI.

## Run locally
```
pip install -r requirements.txt
python app.py
```
'''

for fname, content in [("app.py", app_code), ("requirements.txt", requirements), ("README.md", readme)]:
    with open(f"llm_chatbot_24BAD081/{fname}", "w", encoding="utf-8") as f:
        f.write(content)

print("Project files created:")
for f in sorted(os.listdir("llm_chatbot_24BAD081")):
    print("  -", f, f"({os.path.getsize('llm_chatbot_24BAD081/' + f)} bytes)")
print("\n--- requirements.txt ---")
print(open("llm_chatbot_24BAD081/requirements.txt").read())

In [ ]:
# Download the project folder as a zip to upload to GitHub / Hugging Face Spaces
import shutil
shutil.make_archive("llm_chatbot_24BAD081", "zip", "llm_chatbot_24BAD081")
try:
    from google.colab import files
    files.download("llm_chatbot_24BAD081.zip")
except ImportError:
    print("Zip created: llm_chatbot_24BAD081.zip")

### D.5 Upload the project to GitHub
1. Create a new repository on GitHub, e.g. `llm-chatbot-24BAD081`.
2. Upload `app.py`, `requirements.txt` and `README.md` (**Add file → Upload files**), then commit as `Expt 9 - LLM Chatbot`.
3. Also save this notebook: **File → Save a copy in GitHub**.
4. Add the repository link in your record.

### D.6 Deploy on Hugging Face Spaces (free)
1. Sign in at huggingface.co → **New → Space**.
2. Name: `llm-chatbot-24BAD081`, SDK: **Gradio**, Hardware: **CPU basic (free)**.
3. Upload the same three files (`app.py`, `requirements.txt`, `README.md`) and wait for the build to finish.
4. Open the Space URL; it is the public link of your deployed chatbot. Take a screenshot.

### D.7 Observations and limitations
Fill this in after running with your own outputs:

| Aspect | Observation |
|---|---|
| Response quality | Simple factual, explanatory and coding answers are generally good; a 0.5B model is weaker at maths and long reasoning. |
| Effect of temperature | Low values (0–0.3) give consistent, focused answers; high values (1.0+) give varied, creative but sometimes incoherent text. |
| Max output length | Small limits cut answers off mid-sentence; larger limits increase latency. |
| Conversation history | The bot answers follow-ups (e.g. remembers the user's name) only when history is passed in the prompt; without it, the context is lost. |
| Latency | Faster on GPU than on CPU; latency grows with the number of generated tokens (see the latency table). |
| Hallucinations | The model can confidently invent facts for unknown or non-existent topics (see the hallucination test). |
| Other limitations | Knowledge cut-off, no internet access, small context window, possible bias. |

## RESULT
A Transformer-based LLM (Qwen2.5-0.5B-Instruct) was loaded using Hugging Face Transformers and used to build a chatbot with configurable temperature and output length and conversation history. An interactive chat interface was developed using Gradio, tested for response quality and latency, packaged with `app.py` and `requirements.txt`, and prepared for deployment on GitHub and Hugging Face Spaces.